<a href="https://colab.research.google.com/github/BilalKhaliqWillis/BILAL-Assignment2/blob/main/BILAL_Assignment_9_Machine_Learning_Interoperability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 9: Machine Learning Interoperability


# Part 1: Conceptual Questions

# 1. Define Machine Learning Interoperability
Machine learning interoperability refers to the ability of machine learning models to work across different frameworks, tools, and platforms without compatibility issues.

# Importance
- Enables deployment across multiple platforms.
- Reduces dependency on one vendor.
- Improves portability and flexibility.
- Simplifies collaboration between teams.

---

# 2. Role of ONNX in ML Interoperability
ONNX (Open Neural Network Exchange) is an open-source format for representing machine learning models.

# Benefits
- Framework-independent model sharing.
- Faster deployment.
- Compatibility with many hardware platforms.
- Optimized inference using ONNX Runtime.

---

# 3. Challenges of Vendor Lock-in
Common challenges include:
- Limited portability.
- High migration costs.
- Dependency on proprietary tools.
- Reduced flexibility.

---

# 4. Converting PyTorch Model to ONNX
Steps include:
1. Load trained PyTorch model.
2. Create dummy input tensor.
3. Use torch.onnx.export().
4. Save ONNX model.
5. Verify using ONNX tools.

---

# 5. Importance of Verifying ONNX Models
Verification ensures:
- Model structure is correct.
- No corruption occurred during conversion.
- Compatibility with deployment tools.

# Verification Tools
- onnx.checker
- ONNX Runtime
- Netron visualization tool

---

# 6. ONNX Model Zoo
The ONNX Model Zoo provides pretrained ONNX models.

# Importance
- Easy access to reusable models.
- Supports interoperability.
- Reduces training time.

---

# 7. Purpose of Generic ONNX Checker Script
The generic checker script automates ONNX model validation.

# Benefits
- Saves time.
- Supports batch verification.
- Improves deployment reliability.

---

# 8. Deploying ONNX Models to Azure
Steps:
1. Upload ONNX model.
2. Create Azure ML workspace.
3. Register model.
4. Deploy endpoint.
5. Test inference.

---

# 9. ORT (ONNX Runtime) for Edge Devices
ORT optimizes models by:
- Reducing latency.
- Improving inference speed.
- Supporting hardware acceleration.

---

# 10. Cloud vs Edge Deployment

# Cloud Deployment
- High scalability.
- More computing power.
- Internet required.

# Edge Deployment
- Low latency.
- Works offline.
- Limited hardware resources.


# Part 2: Hands-On Tasks

# Step 1: Install Required Libraries


In [1]:
!pip install tensorflow tf2onnx onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 83.9 MB/s eta 0:00:00


# Step 2: Import Libraries


In [2]:
import tensorflow as tf
import numpy as np
import onnx
import tf2onnx

# Step 3: Load the MNIST Dataset


In [3]:
mnist = tf.keras.datasets.mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train / 255.0
X_test = X_test / 255.0

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Training Data Shape: (60000, 28, 28)
Testing Data Shape: (10000, 28, 28)


# Step 4: Build a Simple LeNet-Style CNN Model


In [5]:
model = tf.keras.models.Sequential([

    tf.keras.Input(shape=(28,28)),

    tf.keras.layers.Reshape((28,28,1)),

    tf.keras.layers.Conv2D(
        6,
        kernel_size=3,
        activation='relu'
    ),

    tf.keras.layers.AveragePooling2D(pool_size=(2,2)),

    tf.keras.layers.Conv2D(
        16,
        kernel_size=3,
        activation='relu'
    ),

    tf.keras.layers.AveragePooling2D(pool_size=(2,2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(120, activation='relu'),

    tf.keras.layers.Dense(84, activation='relu'),

    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape_1 (Reshape)             │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 26, 26, 6)      │            60 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 13, 13, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 11, 11, 16)     │           880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 60,074 (234.66 KB)

 Trainable params: 60,074 (234.66 KB)

 Non-trainable params: 0 (0.00 B)

# Step 5: Train the Model


In [6]:
history = model.fit(
    X_train,
    y_train,
    epochs=1,
    validation_data=(X_test, y_test)
)

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 32s 16ms/step - accuracy: 0.9364 - loss: 0.2149 - val_accuracy: 0.9721 - val_loss: 0.0856


# Step 6: Save TensorFlow Model


In [7]:
model.save("mnist_lenet_model.h5")

print("TensorFlow model saved successfully.")

TensorFlow model saved successfully.


# Step 7: Convert TensorFlow Model to ONNX


In [8]:
!python -m tf2onnx.convert --keras mnist_lenet_model.h5 --output mnist_model.onnx --opset 13

2026-05-09 06:27:57.680416: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-05-09 06:27:57,995 - WARNING - Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
I0000 00:00:1778308078.186090    1603 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1778308078.186372    1603 single_machine.cc:376] Starting new session
I0000 00:00:1778308078.270320    1603 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00:1778308078.270470    1603 single_machine.cc:376] Starting new session
2026-05-09 06:27:58,287 - INFO - Using tensorflow=2.20.0, onnx=1.21.0, tf2onnx=1.17.0/None
2026-05-09 06:27:58,287 - INFO - Using opset <onnx, 13>
2026-05-09 06:27:58,299 - INFO - Computed 0 values for constant foldi

# Step 8: Verify the ONNX Model


In [9]:
onnx_model = onnx.load("mnist_model.onnx")

onnx.checker.check_model(onnx_model)

print("ONNX model verification successful.")

ONNX model verification successful.


# Step 9: Display ONNX Model Graph Summary


In [10]:
print("Model Graph:")
print(onnx_model.graph)

Model Graph:
node {
  input: "args_0:0"
  input: "new_shape__58"
  output: "sequential_1/conv2d_1_1/BiasAdd__22:0"
  name: "sequential_1/conv2d_1_1/BiasAdd__22"
  op_type: "Reshape"
  domain: ""
}
node {
  input: "sequential_1/conv2d_1_1/BiasAdd__22:0"
  input: "sequential_1/conv2d_1_1/convolution/ReadVariableOp:0"
  input: "sequential_1/conv2d_1_1/Squeeze:0"
  output: "sequential_1/conv2d_1_1/BiasAdd:0"
  name: "sequential_1/conv2d_1_1/BiasAdd"
  op_type: "Conv"
  attribute {
    name: "dilations"
    ints: 1
    ints: 1
    type: INTS
  }
  attribute {
    name: "strides"
    ints: 1
    ints: 1
    type: INTS
  }
  attribute {
    name: "kernel_shape"
    ints: 3
    ints: 3
    type: INTS
  }
  attribute {
    name: "group"
    i: 1
    type: INT
  }
  domain: ""
}
node {
  input: "sequential_1/conv2d_1_1/BiasAdd:0"
  output: "sequential_1/conv2d_1_1/Relu:0"
  name: "sequential_1/conv2d_1_1/Relu"
  op_type: "Relu"
}
node {
  input: "sequential_1/conv2d_1_1/Relu:0"
  output: "sequen

# Step 10: Create Generic ONNX Checker Script

In [11]:
%%writefile generic_onnx_checker.py

import sys
import onnx

def verify_model(model_path):
    try:
        model = onnx.load(model_path)
        onnx.checker.check_model(model)

        print(f"Verification successful for: {model_path}")
        print(model.graph)

    except Exception as e:
        print(f"Verification failed for: {model_path}")
        print("Error:", e)

if __name__ == "__main__":
    if len(sys.argv) != 2:
        print("Usage: python generic_onnx_checker.py model.onnx")
    else:
        verify_model(sys.argv[1])

Writing generic_onnx_checker.py


# Step 11: Run Generic ONNX Checker


In [12]:
!python generic_onnx_checker.py mnist_model.onnx

Verification successful for: mnist_model.onnx
node {
  input: "args_0:0"
  input: "new_shape__58"
  output: "sequential_1/conv2d_1_1/BiasAdd__22:0"
  name: "sequential_1/conv2d_1_1/BiasAdd__22"
  op_type: "Reshape"
  domain: ""
}
node {
  input: "sequential_1/conv2d_1_1/BiasAdd__22:0"
  input: "sequential_1/conv2d_1_1/convolution/ReadVariableOp:0"
  input: "sequential_1/conv2d_1_1/Squeeze:0"
  output: "sequential_1/conv2d_1_1/BiasAdd:0"
  name: "sequential_1/conv2d_1_1/BiasAdd"
  op_type: "Conv"
  attribute {
    name: "dilations"
    ints: 1
    ints: 1
    type: INTS
  }
  attribute {
    name: "strides"
    ints: 1
    ints: 1
    type: INTS
  }
  attribute {
    name: "kernel_shape"
    ints: 3
    ints: 3
    type: INTS
  }
  attribute {
    name: "group"
    i: 1
    type: INT
  }
  domain: ""
}
node {
  input: "sequential_1/conv2d_1_1/BiasAdd:0"
  output: "sequential_1/conv2d_1_1/Relu:0"
  name: "sequential_1/conv2d_1_1/Relu"
  op_type: "Relu"
}
node {
  input: "sequential_1/con

# Step 12: Create Enhanced ONNX Checker Script


In [13]:
%%writefile enhanced_checker.py

import sys
import onnx
import logging

logging.basicConfig(
    filename='onnx_verification.log',
    level=logging.INFO,
    format='%(asctime)s - %(message)s'
)

def verify_model(model_path):
    try:
        model = onnx.load(model_path)
        onnx.checker.check_model(model)

        message = f"Verification successful for: {model_path}"
        print(message)
        logging.info(message)

    except Exception as e:
        error_message = f"Verification failed for: {model_path} | Error: {e}"
        print(error_message)
        logging.error(error_message)

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python enhanced_checker.py model1.onnx model2.onnx")
    else:
        for model_path in sys.argv[1:]:
            verify_model(model_path)

Writing enhanced_checker.py


# Step 13: Run Enhanced Checker Script


In [14]:
!python enhanced_checker.py mnist_model.onnx mnist_model.onnx

Verification successful for: mnist_model.onnx
Verification successful for: mnist_model.onnx


# Step 14: View Log File


In [15]:
!cat onnx_verification.log

2026-05-09 06:28:24,486 - Verification successful for: mnist_model.onnx
2026-05-09 06:28:24,487 - Verification successful for: mnist_model.onnx


# Step 15: Conclusion

In this assignment:
- A TensorFlow model was trained using the MNIST dataset.
- The model was converted to ONNX format.
- The ONNX model was verified successfully.
- A reusable ONNX checker script was developed.
- The script was enhanced to support multiple models and logging.

This project demonstrated machine learning interoperability using ONNX.
